# Pristine silicon at glancing incidence

This notebook does one thing: it shows a known glancing electron beam moving across a pristine, unstrained silicon slab. There are no defects, reconstruction variables, validation splits, or saved result files.

In [1]:
%matplotlib widget

from pathlib import Path
import os
import sys

os.environ.setdefault("XLA_PYTHON_CLIENT_MEM_FRACTION", ".2")

# This viewer caches 41 long side views; CPU execution avoids competing with
# reconstruction notebooks for GPU memory. Restart the kernel before running.
os.environ["JAX_PLATFORMS"] = "cpu"

repo_root = Path.cwd()
if not (repo_root / "wide_angle_propagation").is_dir():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

import abtem
from ase import Atoms
from ase.build import bulk
import ipywidgets as widgets
import jax
import jax.numpy as jnp
from IPython.display import display
from matplotlib.colors import PowerNorm
import matplotlib.pyplot as plt
import numpy as np

from wide_angle_propagation.propagation_methods import (
    angular_spectrum_propagation_kernel_1d,
    energy2wavelength,
)
from wide_angle_propagation.ptychography_1d import simulate_glancing_sideview_cache_1d
from wide_angle_propagation.sideview_geometry import make_tilted_gaussian_beam_1d

abtem.config.set({"device": "cpu", "precision": "float64"})
jax.config.update("jax_enable_x64", True)

## Build one pristine silicon slab

A finite diamond-silicon slab is built from explicit atoms and rendered once with the Lobato potential. It is periodic only through the out-of-plane projection width; the surface-normal and propagation directions are finite, so no unit-cell image is cut or stitched at the surface.

In [2]:
energy_eV = 30_000.0
glancing_angle_deg = 2.0
beam_tilt_rad = -np.deg2rad(glancing_angle_deg)
beam_waist_A = 3.0
propagation_length_A = 1000.0
slab_depth_A = 50.0
vacuum_above_A = 40.0
vacuum_below_A = 20.0
si_lattice_A = 5.431

si_unit = bulk("Si", "diamond", a=si_lattice_A, cubic=True)
minimum_projected_si_spacing_A = si_lattice_A / 4.0
surface_clearance_A = minimum_projected_si_spacing_A / 2.0
surface_global_A = vacuum_below_A + slab_depth_A
transverse_extent_A = vacuum_below_A + slab_depth_A + vacuum_above_A

# Build the finite slab from explicit atoms. The repeated bulk coordinates
# point into the material and are then placed below the surface.
n_depth_cells = int(np.ceil(slab_depth_A / si_lattice_A)) + 1
n_length_cells = int(np.ceil(propagation_length_A / si_lattice_A)) + 1
repeated_si = si_unit.repeat((n_depth_cells, 1, n_length_cells))
slab_positions_A = repeated_si.positions
inside_slab = (
    (slab_positions_A[:, 0] <= slab_depth_A - surface_clearance_A + 1e-9)
    & (slab_positions_A[:, 2] < propagation_length_A - 1e-9)
)
slab_positions_A = slab_positions_A[inside_slab].copy()
slab_positions_A[:, 0] = surface_global_A - surface_clearance_A - slab_positions_A[:, 0]
si_slab = Atoms(
    symbols=["Si"] * len(slab_positions_A),
    positions=slab_positions_A,
    cell=(transverse_extent_A, si_lattice_A, propagation_length_A),
    pbc=(False, True, False),
)

builder = abtem.Potential(
    si_slab,
    sampling=(0.115, 0.20),
    slice_thickness=float(si_unit.cell.lengths()[1]),
    projection="finite",
    parametrization="lobato",
    plane="xz",
    periodic=False,
    device="cpu",
)
finite_projected = np.asarray(builder.build(lazy=False).array)[0]
du, ds = (float(value) for value in builder.sampling)
n_u, n_s = finite_projected.shape
s_A = np.arange(n_s) * ds
u_A = np.arange(n_u) * du - surface_global_A
silicon_potential = finite_projected.T / float(si_unit.cell.lengths()[1])

wavelength_A = float(energy2wavelength(energy_eV))
tilted_wave_period_A = wavelength_A / abs(np.sin(beam_tilt_rad))
sampling_diagnostics = {
    "samples across beam waist (u)": beam_waist_A / du,
    "samples per tilted-wave period (u)": tilted_wave_period_A / du,
    "samples per minimum Si spacing (u)": minimum_projected_si_spacing_A / du,
    "samples per minimum Si spacing (s)": minimum_projected_si_spacing_A / ds,
    "transverse Nyquist angle (degrees)": np.rad2deg(np.arcsin(min(1.0, wavelength_A / (2.0 * du)))),
}
assert sampling_diagnostics["samples across beam waist (u)"] >= 20.0
assert sampling_diagnostics["samples per tilted-wave period (u)"] >= 12.0
assert sampling_diagnostics["samples per minimum Si spacing (u)"] >= 10.0
assert sampling_diagnostics["samples per minimum Si spacing (s)"] >= 6.0

print({
    "potential shape": silicon_potential.shape,
    "sampling (ds, du) Å": (ds, du),
    "propagation length Å": float(s_A[-1]),
    "silicon depth Å": slab_depth_A,
    "explicit Si atoms": len(si_slab),
    "outermost atom depth Å": -surface_clearance_A,
    "minimum / maximum potential": (float(silicon_potential.min()), float(silicon_potential.max())),
    "sampling checks": sampling_diagnostics,
})

{'potential shape': (5000, 957), 'sampling (ds, du) Å': (0.2, 0.11494252873563218), 'propagation length Å': 999.8000000000001, 'silicon depth Å': 50.0, 'explicit Si atoms': 13635, 'outermost atom depth Å': -0.678875, 'minimum / maximum potential': (0.0, 119.19617362905802), 'sampling checks': {'samples across beam waist (u)': 26.1, 'samples per tilted-wave period (u)': 17.397961339770358, 'samples per minimum Si spacing (u)': 11.812425000000001, 'samples per minimum Si spacing (s)': 6.788749999999999, 'transverse Nyquist angle (degrees)': 17.673357866679364}}


## Propagate a family of known beams

The slider coordinate is the point at which the nominal beam centre crosses the silicon surface. It covers `s = 400–600 Å` within the full 1000 Å propagation path. Forty-one complete side views are cached once, so moving the slider does not rerun the propagation. The fine simulation grid is downsampled only when storing the display cache.

In [3]:
scan_coordinates_A = np.linspace(400.0, 600.0, 201)
beam_centres_at_entrance_A = -scan_coordinates_A * np.tan(beam_tilt_rad)
input_probes = jnp.stack([
    make_tilted_gaussian_beam_1d(
        jnp.asarray(u_A),
        energy_eV,
        waist=beam_waist_A,
        center=float(center_A),
        tilt=beam_tilt_rad,
    )
    for center_A in beam_centres_at_entrance_A
]).astype(jnp.complex128)
kernel = angular_spectrum_propagation_kernel_1d(n_u, du, ds, energy_eV)
window_starts = jnp.zeros(len(scan_coordinates_A), dtype=jnp.int32)

sideviews = simulate_glancing_sideview_cache_1d(
    jnp.asarray(silicon_potential),
    input_probes,
    window_starts,
    n_s,
    kernel,
    ds,
    energy_eV,
    jnp.arange(len(scan_coordinates_A)),
    transverse_coordinates=jnp.asarray(u_A),
    scan_coordinates=jnp.asarray(scan_coordinates_A),
    axial_stride=5,
    transverse_stride=2,
    metadata={"specimen": "pristine silicon", "glancing_angle_deg": glancing_angle_deg},
)

wavelength_A = float(energy2wavelength(energy_eV))
detector_frequencies = np.fft.fftshift(np.fft.fftfreq(n_u, d=du))
detector_angles_mrad = 1e3 * np.arcsin(np.clip(wavelength_A * detector_frequencies, -1.0, 1.0))
print(f"Cached {len(scan_coordinates_A)} pristine-silicon side views.")

Cached 201 pristine-silicon side views.


## Interactive side viewer

Move the slider to change the beam landing position. The left panel overlays propagated intensity on the fixed silicon potential. The right panels show the positive-`u` exit intensity, its unwrapped phase where the wave is illuminated, and the positive-angle detector intensity.

In [4]:
side_s_A = np.asarray(sideviews.local_s_coordinates)
side_u_A = np.asarray(sideviews.sideview_u_coordinates)
potential_extent = [s_A[0] - ds / 2.0, s_A[-1] + ds / 2.0, u_A[0] - du / 2.0, u_A[-1] + du / 2.0]
side_ds_A = float(side_s_A[1] - side_s_A[0])
side_du_A = float(side_u_A[1] - side_u_A[0])
sideview_extent = [side_s_A[0] - side_ds_A / 2.0, side_s_A[-1] + side_ds_A / 2.0, side_u_A[0] - side_du_A / 2.0, side_u_A[-1] + side_du_A / 2.0]
positive_exit = (u_A >= 0.0) & (u_A <= 35.0)
positive_exit_intensity_max = float(np.max(np.abs(np.asarray(sideviews.exit_waves)[:, positive_exit]) ** 2))
potential_max = float(np.max(silicon_potential))
beam_display_dynamic_range = 1e5

with plt.ioff():
    figure = plt.figure(figsize=(14, 8.0), constrained_layout=True)

def draw_scan(scan_index):
    scan_index = int(scan_index)
    figure.clear()
    grid = figure.add_gridspec(3, 2, width_ratios=(2.3, 1.0))
    side_ax = figure.add_subplot(grid[:, 0])
    exit_ax = figure.add_subplot(grid[0, 1])
    phase_ax = figure.add_subplot(grid[1, 1])
    detector_ax = figure.add_subplot(grid[2, 1])

    side_ax.set_facecolor("#090c12")
    side_ax.axhspan(-slab_depth_A, 0.0, color="#313846", alpha=0.55, zorder=-1, label="Si slab")
    potential_cmap = plt.get_cmap("magma").copy()
    potential_cmap.set_under(alpha=0.0)
    side_ax.imshow(
        silicon_potential.T,
        origin="lower",
        aspect="auto",
        extent=potential_extent,
        cmap=potential_cmap,
        norm=PowerNorm(gamma=0.45, vmin=max(potential_max * 1e-4, 1e-12), vmax=potential_max),
        alpha=0.62,
        interpolation="hanning",
    )
    scan_intensity = np.asarray(sideviews.sideview_intensities[scan_index]).T
    normalized_intensity = scan_intensity / (float(scan_intensity.max()) + 1e-30)
    # Compress the image values so intensities down to about 1e-5 of the
    # incident peak remain visible. This is a display transform, not a log axis.
    beam_display = np.log1p(beam_display_dynamic_range * normalized_intensity) / np.log1p(beam_display_dynamic_range)
    beam_alpha = 0.94 * np.sqrt(beam_display)
    intensity_image = side_ax.imshow(
        beam_display,
        origin="lower",
        aspect="auto",
        extent=sideview_extent,
        cmap="viridis",
        vmin=0.0,
        vmax=1.0,
        alpha=beam_alpha,
        interpolation="hanning",
    )
    landing_A = float(scan_coordinates_A[scan_index])
    centreline_A = np.tan(beam_tilt_rad) * (side_s_A - landing_A)
    side_ax.plot(side_s_A, centreline_A, "w--", linewidth=1.2, label="nominal beam centre")
    side_ax.scatter([landing_A], [0.0], color="cyan", edgecolor="black", s=45, zorder=5, label="surface landing")
    side_ax.set(xlim=(s_A[0], s_A[-1]), ylim=(-55.0, 35.0), xlabel="s (Å)", ylabel="u (Å)", title=f"Pristine Si; landing at s = {landing_A:.1f} Å")
    side_ax.legend(loc="lower left")
    figure.colorbar(intensity_image, ax=side_ax, label="log-compressed normalized beam intensity")

    exit_wave = np.asarray(sideviews.exit_waves[scan_index])
    positive_exit_wave = exit_wave[positive_exit]
    positive_exit_u_A = u_A[positive_exit]
    positive_exit_intensity = np.abs(positive_exit_wave) ** 2
    exit_ax.plot(positive_exit_u_A, positive_exit_intensity)
    exit_ax.set(xlim=(0.0, 35.0), ylim=(0.0, positive_exit_intensity_max * 1.03), xlabel="u (Å)", ylabel="intensity", title="Positive-u exit intensity")

    phase_is_valid = positive_exit_intensity > max(float(positive_exit_intensity.max()) * 1e-5, 1e-30)
    phase_u_A = positive_exit_u_A[phase_is_valid]
    unwrapped_phase = np.unwrap(np.angle(positive_exit_wave[phase_is_valid]))
    phase_ax.plot(phase_u_A, unwrapped_phase, linewidth=1.0)
    phase_ax.set(xlim=(0.0, 35.0), xlabel="u (Å)", ylabel="phase (rad)", title="Unwrapped exit-wave phase")

    positive_angles = detector_angles_mrad > 0.0
    positive_detector_intensity = np.asarray(sideviews.detector_intensities[scan_index])[positive_angles]
    detector_ax.plot(detector_angles_mrad[positive_angles], positive_detector_intensity, linewidth=1.2)
    detector_ax.axvline(-1e3 * beam_tilt_rad, color="tab:red", linestyle="--", linewidth=0.9, label="specular")
    detector_ax.set(xlim=(0.0, 80.0), xlabel="positive detector angle (mrad)", ylabel="intensity", title="Positive-angle far field (linear scale)")
    detector_ax.set_ylim(bottom=0.0)
    detector_ax.legend(fontsize=8, frameon=False)
    figure.canvas.draw_idle()

slider = widgets.SelectionSlider(
    options=[(f"{index:02d} — {landing:6.1f} Å", index) for index, landing in enumerate(scan_coordinates_A)],
    value=len(scan_coordinates_A) // 2,
    description="landing position",
    continuous_update=False,
    style={"description_width": "initial"},
    layout=widgets.Layout(width="720px"),
)
slider.observe(lambda change: draw_scan(change["new"]), names="value")
draw_scan(slider.value)
display(widgets.VBox([slider, figure.canvas]))